# 3D Spatial Cluster Comparison

Side-by-side comparison of LRG cluster assignments between pre/post phases
in 3D anatomical space.

In [ ]:
%matplotlib inline
from lrgsglib.config.funcs import move_to_rootf
move_to_rootf(pathname="lrgeegfc")
from lrg_eegfc.notebook import *

## Parameters

In [ ]:
patient = "Pat_02"
phase_a = "rsPre"
phase_b = "rsPost"
band = "beta"
fc_method = "msc"

## Side-by-Side 3D Comparison

Compare cluster organization before and after intervention.
Both views share the same camera angle for direct comparison.

In [ ]:
from lrg_eegfc.visuals import plot_spatial_clusters_comparison

fig = plot_spatial_clusters_comparison(
    patient,
    phase_a,
    phase_b,
    band,
    fc_method=fc_method,
    node_size=10,
)
fig.show()

## Compare Multiple Bands

In [ ]:
from lrg_eegfc.config.const import BRAIN_BANDS_NAMES

for band_name in ["alpha", "beta", "gamma"]:
    try:
        fig = plot_spatial_clusters_comparison(
            patient,
            phase_a,
            phase_b,
            band_name,
            fc_method=fc_method,
            node_size=8,
            height=500,
        )
        fig.show()
    except FileNotFoundError as e:
        print(f"Skipping {band_name}: {e}")

## Detailed Reorganization Analysis

Load cluster labels directly for further analysis.

In [ ]:
from pathlib import Path
from scipy.cluster.hierarchy import fcluster

from lrg_eegfc.utils.io.patient import load_patient_metadata
from lrg_eegfc.workflow.lrg import load_lrg_result
from lrg_eegfc.visuals import prepare_spatial_coordinates

# Load data
metadata = load_patient_metadata(patient, Path("data/stereoeeg_patients"))
result_a = load_lrg_result(patient, phase_a, band, fc_method, Path("data/lrg_cache"))
result_b = load_lrg_result(patient, phase_b, band, fc_method, Path("data/lrg_cache"))

labels_a = fcluster(result_a.linkage_matrix, result_a.optimal_threshold, criterion="distance")
labels_b = fcluster(result_b.linkage_matrix, result_b.optimal_threshold, criterion="distance")

print(f"Clusters in {phase_a}: {len(np.unique(labels_a))}")
print(f"Clusters in {phase_b}: {len(np.unique(labels_b))}")
print(f"Nodes that changed cluster: {np.sum(labels_a != labels_b)}")

In [ ]:
# Identify which electrodes changed clusters
changed_mask = labels_a != labels_b
changed_channels = metadata["label"][changed_mask].tolist()

print(f"\nElectrodes that changed cluster:")
for ch, old_c, new_c in zip(
    metadata["label"][changed_mask],
    labels_a[changed_mask],
    labels_b[changed_mask]
):
    print(f"  {ch}: {old_c} -> {new_c}")

## Export Comparison HTML

In [ ]:
html_dir = Path("data/figures/spatial") / patient
html_dir.mkdir(parents=True, exist_ok=True)
html_path = html_dir / f"{band}_{phase_a}_vs_{phase_b}_{fc_method}_comparison.html"

fig.write_html(html_path)
print(f"Saved: {html_path}")